# Directorio de Establecimientos de Reclusión del Orden Nacional

Fuente: [datos.gov.co — 24zf-4cfu](https://www.datos.gov.co/Justicia-y-Derecho/Directorio-de-Establecimientos-de-Reclusi-n-de-Ord/24zf-4cfu/about_data) (API Socrata SODA).

In [ ]:
import re
import requests
import pandas as pd

URL = "https://www.datos.gov.co/resource/24zf-4cfu.json"

In [ ]:
def traer_datos(url: str, page_size: int = 1000) -> pd.DataFrame:
    """Descarga todos los registros paginando con $limit/$offset."""
    registros, offset = [], 0
    while True:
        r = requests.get(url, params={"$limit": page_size, "$offset": offset, "$order": ":id"}, timeout=60)
        r.raise_for_status()
        lote = r.json()
        registros.extend(lote)
        if len(lote) < page_size:
            break
        offset += page_size
    return pd.DataFrame(registros)

df_raw = traer_datos(URL)
print(df_raw.shape)
df_raw.head()

## Limpieza
Nombres de columnas legibles y coordenadas convertidas de grados-minutos-segundos (texto con formatos variados) a decimales.

In [ ]:
def dms_a_decimal(valor):
    """Convierte textos como -04°12'35.748\" o 05° 53' 30,96'' a grados decimales."""
    if pd.isna(valor):
        return None
    s = str(valor).strip().replace(",", ".")
    negativo = s.startswith("-") or s.upper().endswith(("S", "W", "O"))
    nums = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", s)]
    if not nums:
        return None
    grados = nums[0] + (nums[1] / 60 if len(nums) > 1 else 0) + (nums[2] / 3600 if len(nums) > 2 else 0)
    return -grados if negativo else grados

df = df_raw.rename(columns={
    "direcci_n": "direccion",
    "clasificaci_n_art_11_ley": "clasificacion_art_11_ley_65",
    "nombre_establecimiento_sisipec": "nombre_establecimiento",
})
df["latitud"] = df["latitud"].apply(dms_a_decimal).round(6)
df["longitud"] = df["longitud"].apply(dms_a_decimal).round(6)
for c in df.select_dtypes(exclude="number"):
    df[c] = df[c].str.strip()

df.info()
df.head()

In [ ]:
# Coordenadas fuera del rango aproximado de Colombia (revisar manualmente)
fuera = df[~df["latitud"].between(-5, 14) | ~df["longitud"].between(-82, -66)]
fuera[["nombre_establecimiento", "latitud", "longitud"]]

In [ ]:
df.to_csv("establecimientos_reclusion.csv", index=False, encoding="utf-8-sig")

## Otras cárceles y policía (OpenStreetMap)

La base del INPEC solo incluye establecimientos del orden nacional. Para cárceles municipales/departamentales y sitios de policía (estaciones, subestaciones, CAI) no hay una fuente oficial nacional abierta, así que se toman de OpenStreetMap vía la API Overpass:

- `amenity=prison` → cárceles (incluye también las del INPEC).
- `amenity=police` → estaciones, CAI, comandos, etc.

**Ojo:** OSM es colaborativo; la cobertura es buena en ciudades y desigual en zonas rurales, y muchos registros no tienen nombre.

### Enlaces

- OpenStreetMap: https://www.openstreetmap.org
- API Overpass usada en el notebook: https://overpass-api.de/api/interpreter (respaldo: https://overpass.kumi.systems/api/interpreter)
- Ver la consulta en un mapa: https://overpass-turbo.eu (pegar la consulta de `traer_osm` y pulsar **Run**)
- Definición de las etiquetas:
  - Cárceles: https://wiki.openstreetmap.org/wiki/Tag:amenity%3Dprison
  - Policía: https://wiki.openstreetmap.org/wiki/Tag:amenity%3Dpolice
- Ver un lugar puntual: `https://www.openstreetmap.org/<id_fuente>`, por ejemplo https://www.openstreetmap.org/way/123456
- Licencia (ODbL): https://www.openstreetmap.org/copyright. Al publicar estos datos hay que citar "© colaboradores de OpenStreetMap" y compartir la tabla derivada con la misma licencia.


In [ ]:
import numpy as np

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

def traer_osm(amenity: str) -> pd.DataFrame:
    """Descarga de OpenStreetMap todos los elementos amenity=<amenity> dentro de Colombia."""
    query = f"""
    [out:json][timeout:180];
    area["ISO3166-1"="CO"][admin_level=2]->.co;
    nwr["amenity"="{amenity}"](area.co);
    out center tags;
    """
    ultimo_error = None
    for url in OVERPASS_URLS:
        try:
            r = requests.post(url, data={"data": query}, headers={"User-Agent": "query-collection-notebook"}, timeout=240)
            r.raise_for_status()
            elementos = r.json()["elements"]
            break
        except Exception as e:  # probar el siguiente servidor
            ultimo_error = e
    else:
        raise RuntimeError(f"No se pudo consultar Overpass: {ultimo_error}")

    filas = []
    for e in elementos:
        tags = e.get("tags", {})
        centro = e.get("center", {})
        filas.append({
            "osm_id": f"{e['type']}/{e['id']}",
            "amenity": amenity,
            "nombre": tags.get("name"),
            "operador": tags.get("operator"),
            "ciudad": tags.get("addr:city"),
            "latitud": e.get("lat", centro.get("lat")),
            "longitud": e.get("lon", centro.get("lon")),
        })
    return pd.DataFrame(filas)

osm_carceles = traer_osm("prison")
osm_policia = traer_osm("police")
print("Cárceles OSM:", len(osm_carceles), "| Policía OSM:", len(osm_policia))
osm_carceles.head()

### Separar cárceles del INPEC de las demás

Una cárcel de OSM se considera del INPEC si está a menos de `RADIO_INPEC_KM` de un establecimiento de la base oficial; las demás quedan como "otra cárcel" (municipales, departamentales, de menores, etc.).

In [ ]:
def distancia_km(lat1, lon1, lat2, lon2):
    """Haversine vectorizado: matriz de distancias (len(lat1) x len(lat2)) en km."""
    lat1, lon1, lat2, lon2 = map(np.radians, (np.asarray(lat1)[:, None], np.asarray(lon1)[:, None],
                                              np.asarray(lat2)[None, :], np.asarray(lon2)[None, :]))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

RADIO_INPEC_KM = 1.0

d = distancia_km(osm_carceles["latitud"], osm_carceles["longitud"], df["latitud"], df["longitud"])
osm_carceles["dist_inpec_km"] = d.min(axis=1).round(3)
osm_carceles["inpec_mas_cercano"] = df["nombre_establecimiento"].to_numpy()[d.argmin(axis=1)]
osm_carceles["es_inpec"] = osm_carceles["dist_inpec_km"] <= RADIO_INPEC_KM

print(osm_carceles["es_inpec"].value_counts())
otras_carceles = osm_carceles[~osm_carceles["es_inpec"]]
otras_carceles[["nombre", "operador", "ciudad", "latitud", "longitud", "dist_inpec_km"]].head(20)

### Tabla unificada

Una fila por lugar con `tipo` ∈ {`INPEC`, `Otra cárcel`, `Policía`}, lista para cruzar por distancia con otras bases (p. ej. transacciones).

In [ ]:
lugares = pd.concat([
    pd.DataFrame({
        "tipo": "INPEC",
        "fuente": "datos.gov.co 24zf-4cfu",
        "id_fuente": df["cod_establecimiento"],
        "nombre": df["nombre_establecimiento"],
        "departamento": df["departamento"],
        "ciudad": df["ciudad"],
        "latitud": df["latitud"],
        "longitud": df["longitud"],
    }),
    pd.DataFrame({
        "tipo": "Otra cárcel",
        "fuente": "OpenStreetMap",
        "id_fuente": otras_carceles["osm_id"],
        "nombre": otras_carceles["nombre"],
        "departamento": None,
        "ciudad": otras_carceles["ciudad"],
        "latitud": otras_carceles["latitud"],
        "longitud": otras_carceles["longitud"],
    }),
    pd.DataFrame({
        "tipo": "Policía",
        "fuente": "OpenStreetMap",
        "id_fuente": osm_policia["osm_id"],
        "nombre": osm_policia["nombre"],
        "departamento": None,
        "ciudad": osm_policia["ciudad"],
        "latitud": osm_policia["latitud"],
        "longitud": osm_policia["longitud"],
    }),
], ignore_index=True)

print(lugares["tipo"].value_counts())
lugares.to_csv("lugares_reclusion_policia.csv", index=False, encoding="utf-8-sig")
lugares.sample(10, random_state=1)

### Base solo de cárceles (INPEC + otras cárceles)

Sin sitios de policía. Es la base recomendada para cruzar con transacciones: el INPEC como fuente principal y las otras cárceles de OpenStreetMap marcadas aparte (revisarlas manualmente antes de usarlas).

In [ ]:
carceles = lugares[lugares["tipo"].isin(["INPEC", "Otra cárcel"])].reset_index(drop=True)

print(carceles["tipo"].value_counts())
carceles.to_csv("carceles_inpec_otras.csv", index=False, encoding="utf-8-sig")
carceles.head()